# IGF1R / calx-β — binding inspection (Notebook 3)

For each replicate, compare per-frame MM-GBSA ΔG (Notebook 2, primary GB model) with calx-β geometry from Notebook 1 at exactly the same frames.

**Question:** when ΔG ≥ 0, does the geometry stay inside the range seen in favorable-ΔG frames (an energy fluctuation), or does it leave that range (the pose weakening or being lost)?

**Geometry:** three measures from Notebook 1:
- calx-β RMSD after fitting on the receptor backbone, against the docked pose;
- the number of receptor contact residues;
- Q, the fraction of docked residue contacts kept.

**Inputs per replicate:**
- `rep{n}/md_binding_metrics.csv` (Notebook 1);
- `rep{n}/mmpbsa/frame_times.csv` and `rep{n}/mmpbsa/{model}/delta_g_per_frame.csv` (Notebook 2).

Frames are joined on their exact production time, and any frame without a match stops the notebook.

**Outputs:**
- `rep{n}/binding_inspection_unfavorable_frames.csv`;
- `rep{n}/binding_inspection_summary.json`;
- `binding_inspection_summary.json` across replicates.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_data_dir() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "md_calx_3qqu_full").is_dir():
            return candidate
        if (candidate / "data" / "md_calx_3qqu_full").is_dir():
            return candidate / "data"
    raise FileNotFoundError("Could not find data/md_calx_3qqu_full — run Notebooks 1 and 2 first.")


DATA_DIR = find_data_dir()
MD_ROOT = DATA_DIR / "md_calx_3qqu_full"
MODEL = "gbn2"  # Notebook 2's PRIMARY_MODEL
REPLICATES: list[int] | None = None  # None = every replicate with Notebook 1 and 2 results
TIME_TOLERANCE_PS = 1e-3  # MM-GBSA and Notebook 1 frames must share the same production time

RMSD_COL = "calx_rmsd_receptor_fit_nm"
print(f"MD_ROOT={MD_ROOT}  model={MODEL}")

In [ ]:
def load_replicate(rep_dir: Path) -> pd.DataFrame:
    """Join Notebook 2 per-frame ΔG to Notebook 1 geometry on exact production time."""
    binding = pd.read_csv(rep_dir / "md_binding_metrics.csv")
    need = {"time_ps", RMSD_COL, "receptor_contact_residues", "q_native"}
    missing = need - set(binding.columns)
    if missing:
        raise ValueError(f"{rep_dir.name}/md_binding_metrics.csv lacks {sorted(missing)} — re-run Notebook 1 analysis")
    delta = pd.read_csv(rep_dir / "mmpbsa" / MODEL / "delta_g_per_frame.csv")

    t_bind = binding["time_ps"].to_numpy(dtype=float)
    idx = np.searchsorted(t_bind, delta["time_ps"].to_numpy(dtype=float))
    idx = np.clip(idx, 0, len(t_bind) - 1)
    err = np.abs(t_bind[idx] - delta["time_ps"].to_numpy(dtype=float))
    for i in np.where(err > TIME_TOLERANCE_PS)[0]:  # neighbour on the left may be the exact match
        if idx[i] > 0 and abs(t_bind[idx[i] - 1] - delta["time_ps"].iloc[i]) <= TIME_TOLERANCE_PS:
            idx[i] -= 1
    err = np.abs(t_bind[idx] - delta["time_ps"].to_numpy(dtype=float))
    if np.any(err > TIME_TOLERANCE_PS):
        bad = delta["time_ps"][err > TIME_TOLERANCE_PS].head(5).tolist()
        raise ValueError(
            f"{rep_dir.name}: MM-GBSA frames at t = {bad} ps have no Notebook 1 frame at the same time. "
            "Re-run Notebook 1 analysis and Notebook 2 on the same md_pbc.xtc."
        )
    geom = binding.iloc[idx][[RMSD_COL, "receptor_contact_residues", "q_native"]].reset_index(drop=True)
    joined = pd.concat([delta.reset_index(drop=True), geom], axis=1)
    joined["unfavorable"] = joined["delta_g_kcal"] >= 0.0
    return joined


reps: list[int] = []
DATA: dict[int, pd.DataFrame] = {}
for rep_dir in sorted(MD_ROOT.glob("rep*"), key=lambda p: int(p.name[3:]) if p.name[3:].isdigit() else 0):
    if not rep_dir.name[3:].isdigit():
        continue
    rep = int(rep_dir.name[3:])
    if REPLICATES is not None and rep not in REPLICATES:
        continue
    if not (rep_dir / "mmpbsa" / MODEL / "delta_g_per_frame.csv").exists():
        print(f"rep{rep}: no Notebook 2 results for {MODEL} — skipped")
        continue
    DATA[rep] = load_replicate(rep_dir)
    d = DATA[rep]
    print(
        f"rep{rep}: {len(d)} frames ({d['time_ps'].iloc[0]:.0f}–{d['time_ps'].iloc[-1]:.0f} ps), "
        f"ΔG ≥ 0 in {int(d['unfavorable'].sum())}; all frames matched exactly on time"
    )
if not DATA:
    raise FileNotFoundError("No replicate has both Notebook 1 metrics and Notebook 2 per-frame ΔG.")

In [ ]:
def geometry_bounds_from_favorable(joined: pd.DataFrame) -> dict[str, float | int | str]:
    """Bound-like envelope = range of geometry seen in favorable-ΔG (ΔG < 0) frames of the same replicate.

    RMSD ≤ the favorable maximum, contact residues ≥ the favorable minimum, Q ≥ the favorable minimum.
    With fewer than two favorable frames, the full series is used.
    """
    fav = joined.loc[~joined["unfavorable"]]
    ref, note = (fav, "favorable_delta_g_ensemble") if len(fav) >= 2 else (joined, "full_series_fallback")
    return {
        "rmsd_max_nm": float(ref[RMSD_COL].max()),
        "contacts_min": float(ref["receptor_contact_residues"].min()),
        "q_min": float(ref["q_native"].min()),
        "n_reference_frames": int(len(ref)),
        "reference": note,
    }


def classify_row(row: pd.Series, b: dict) -> str:
    if float(row["delta_g_kcal"]) < 0.0:
        return "favorable"
    like_bound = (
        float(row[RMSD_COL]) <= b["rmsd_max_nm"]
        and float(row["receptor_contact_residues"]) >= b["contacts_min"]
        and float(row["q_native"]) >= b["q_min"]
    )
    return "unfavorable_energy_noise" if like_bound else "unfavorable_geometry_weak"


BOUNDS: dict[int, dict] = {}
for rep, joined in DATA.items():
    BOUNDS[rep] = b = geometry_bounds_from_favorable(joined)
    joined["class"] = joined.apply(lambda r: classify_row(r, b), axis=1)
    print(
        f"rep{rep} envelope ({b['reference']}, n={b['n_reference_frames']}): "
        f"RMSD ≤ {b['rmsd_max_nm']:.3f} nm, contact residues ≥ {b['contacts_min']:.0f}, Q ≥ {b['q_min']:.2f}"
    )

In [ ]:
for rep, joined in DATA.items():
    b = BOUNDS[rep]
    bad = joined[joined["unfavorable"]]
    t_ns = joined["time_ps"] / 1000.0
    fig, axes = plt.subplots(4, 1, figsize=(9, 9), sharex=True)
    panels = [
        ("delta_g_kcal", "ΔG (kcal/mol)", "#377eb8", None),
        (RMSD_COL, "Calx-β RMSD,\nreceptor fit (nm)", "#4daf4a", ("rmsd_max_nm", "favorable max")),
        ("receptor_contact_residues", "Contact residues", "#984ea3", ("contacts_min", "favorable min")),
        ("q_native", "Fraction of docked\ncontacts (Q)", "#ff7f00", ("q_min", "favorable min")),
    ]
    for ax, (col, ylabel, color, bound) in zip(axes, panels):
        ax.plot(t_ns, joined[col], lw=1.0, color=color)
        if not bad.empty:
            ax.scatter(bad["time_ps"] / 1000.0, bad[col], s=16, c="#e41a1c", zorder=3, label="ΔG ≥ 0")
        if bound:
            ax.axhline(b[bound[0]], color="0.3", ls="--", lw=1, label=f"{bound[1]} = {b[bound[0]]:.2f}")
        ax.set_ylabel(ylabel)
        if bound or not bad.empty:
            ax.legend(loc="best", fontsize=7)
    axes[0].axhline(0.0, color="0.4", ls=":", lw=1)
    axes[0].set_title(f"Replicate {rep}: MM-GBSA ΔG ({MODEL}) with Notebook 1 geometry")
    axes[-1].set_xlabel("Production time (ns)")
    plt.tight_layout()
    plt.show()
    plt.close("all")

In [ ]:
rows = []
for rep, joined in DATA.items():
    counts = joined["class"].value_counts()
    rows.append(
        {
            "replicate": rep,
            "frames": len(joined),
            "dg_ge_0": int(joined["unfavorable"].sum()),
            "energy_noise": int(counts.get("unfavorable_energy_noise", 0)),
            "geometry_weak": int(counts.get("unfavorable_geometry_weak", 0)),
        }
    )
    unfav = joined[joined["unfavorable"]]
    if len(unfav):
        cols = ["mmpbsa_frame", "time_ps", "delta_g_kcal", RMSD_COL, "receptor_contact_residues", "q_native", "class"]
        display(Markdown(f"#### Replicate {rep}: frames with ΔG ≥ 0 ({len(unfav)} / {len(joined)})"))
        display(unfav[cols].reset_index(drop=True).round(3))
CLASS_TABLE = pd.DataFrame(rows)
display(Markdown("**Frames with ΔG ≥ 0, by replicate:**"))
display(CLASS_TABLE)

In [ ]:
def replicate_reading(rep: int, joined: pd.DataFrame, b: dict) -> tuple[str, str]:
    """(verdict, one-paragraph reading) for one replicate."""
    n, unfav = len(joined), joined[joined["unfavorable"]]
    n_bad = len(unfav)
    n_noise = int((unfav["class"] == "unfavorable_energy_noise").sum())
    n_geo = n_bad - n_noise
    text = (
        f"**Replicate {rep}:** {n} frames, mean ΔG {joined['delta_g_kcal'].mean():.1f} kcal/mol; "
        f"median calx-β RMSD (receptor fit) {joined[RMSD_COL].median():.2f} nm, "
        f"median Q {joined['q_native'].median():.2f}. "
    )
    if n_bad == 0:
        return "consistent", text + "No frame has ΔG ≥ 0."
    late = int((unfav["time_ps"] >= joined["time_ps"].quantile(0.75)).sum())
    text += (
        f"{n_bad} frames have ΔG ≥ 0: {n_noise} stay inside the favorable-frame geometry "
        f"(energy fluctuation), {n_geo} fall outside it; {late} of them are in the last quarter of the run."
    )
    return ("consistent" if n_noise >= n_geo else "geometry_weakens"), text


verdicts, paragraphs = {}, []
for rep, joined in DATA.items():
    verdicts[rep], para = replicate_reading(rep, joined, BOUNDS[rep])
    paragraphs.append(para)

n_reps = len(DATA)
n_consistent = sum(v == "consistent" for v in verdicts.values())
n_clean = sum(int(j["unfavorable"].sum()) == 0 for j in DATA.values())
if n_clean == n_reps:
    overall = f"No frame has ΔG ≥ 0 in any of the **{n_reps}** replicate(s), so there is nothing to classify."
else:
    overall = (
        f"In **{n_consistent} of {n_reps}** replicate(s), ΔG stays negative or the frames with ΔG ≥ 0 are mostly "
        "energy fluctuations inside the geometry of the favorable frames — consistent with a retained docked "
        "pose, not with dissociation."
    )
if n_consistent < n_reps:
    weak = [str(r) for r, v in verdicts.items() if v != "consistent"]
    overall += (
        f" In replicate(s) {', '.join(weak)}, unfavorable ΔG coincides with the pose leaving that geometry; "
        "report these as replicates where the docked pose weakens."
    )
caveat = (
    "The envelope is the geometry range of the same replicate's favorable frames, so this test asks whether "
    "unfavorable frames look different, not whether the pose is correct. MM-GBSA values are estimates with "
    "systematic errors of tens of kcal/mol (Notebook 2), not experimental binding free energies."
)
interpretation = "\n\n".join(paragraphs + [f"**Overall:** {overall}", caveat])
display(Markdown("## Interpretation and conclusion\n\n" + interpretation))

for rep, joined in DATA.items():
    rep_dir = MD_ROOT / f"rep{rep}"
    joined[joined["unfavorable"]].to_csv(rep_dir / "binding_inspection_unfavorable_frames.csv", index=False)
    (rep_dir / "binding_inspection_summary.json").write_text(
        json.dumps(
            {
                "replicate": rep,
                "model": MODEL,
                "n_frames": len(joined),
                "class_counts": {k: int(v) for k, v in joined["class"].value_counts().items()},
                "geometry_bounds": BOUNDS[rep],
                "verdict": verdicts[rep],
                "mean_delta_g_kcal": float(joined["delta_g_kcal"].mean()),
                "median_calx_rmsd_receptor_fit_nm": float(joined[RMSD_COL].median()),
                "median_q_native": float(joined["q_native"].median()),
            },
            indent=2,
        )
        + "\n"
    )
summary_path = MD_ROOT / "binding_inspection_summary.json"
summary_path.write_text(
    json.dumps(
        {"model": MODEL, "per_replicate": CLASS_TABLE.to_dict(orient="records"),
         "verdicts": {str(k): v for k, v in verdicts.items()}, "interpretation": interpretation},
        indent=2,
    )
    + "\n"
)
print(f"Wrote per-replicate files and {summary_path}")

In [ ]:
! jupyter nbconvert --to html --no-input --embed-images md3_igf1r3qqu_calxbeta_binding_inspection.ipynb